# Intent Classifier — Baseline vs. ML (+ twist: synthetic ES/PT data)
### Factored AI & Data Hackathon 2026 — Team DataMastersGT

Same approach as the priority notebook: we first document why the direct approach did not
work, before showing the solution.

## Step 1 — What we tried first (and why it did not work)

The hypothesis: train an intent classifier (`main_topics`: Transactional, Product, Complaint,
Technical, Commercial, Retention) on the real text of `call_transcripts.customer_text`, compared
against a keyword-matching baseline.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
base = "../hackathon-data"
con.execute(f"CREATE VIEW transcripts AS SELECT * FROM read_csv_auto('{base}/call_transcripts/**/*.csv', ignore_errors=true, union_by_name=true)")
df_real = con.execute("SELECT customer_text, main_topics FROM transcripts WHERE customer_text IS NOT NULL LIMIT 20").df()
for _, r in df_real.head(8).iterrows():
    print(r["main_topics"], "|", r["customer_text"][:90])

Técnico | Hola, buenos días. Quisiera saber cuál es mi saldo actual en mi cuenta de ahorros. Muy bie
Transaccional | Buenas tardes, necesito consultar el saldo de mi tarjeta de crédito. ¿Y eso cuánto tiempo 
Transaccional | Hola, buenos días. Quisiera saber cuál es mi saldo actual en mi cuenta de ahorros.
Comercial | Buenas tardes, necesito consultar el saldo de mi tarjeta de crédito.
Transaccional | Buenas tardes, necesito consultar el saldo de mi tarjeta de crédito.
Producto | Buenas tardes, necesito consultar el saldo de mi tarjeta de crédito.
Transaccional | Hola, buenos días. Quisiera saber cuál es mi saldo actual en mi cuenta de ahorros. Muy bie
Producto | Buenas tardes, necesito consultar el saldo de mi tarjeta de crédito.


**Result:** the text is the same generic template repeated ("Hola, buenos días. Quisiera saber cuál es mi saldo...")
across completely different categories — there is no real text to classify; the synthetic dataset did not generate
conversations tied to its own label (macro F1 on that text: keywords 0.087, TF-IDF+LogReg 0.107, i.e. random over 6 classes).

## Step 2 — v1 of our own corpus, and why we threw it away

Our first own corpus (v1) had **144 phrases** from ~6 templates per class, with a *random* train/test split.
Re-running it gave: **keywords 0.749 vs ML 0.745 macro F1** (ML did not beat the baseline) and **ES 0.861 vs PT 0.657**.
Two problems: (1) too few templates, so keywords already cover them; (2) the random split puts near-copies of the
same template in train *and* test (leakage), so the number does not measure generalization at all.

## Step 3 — v2 corpus + evaluation split by template (no leakage)

- ~20 templates per class **per language** (ES and PT balanced), including "hard" phrasings without the obvious
  keyword (e.g. *"aparece un movimiento raro en mi estado de cuenta"*).
- Realistic noise: greetings, lowercase, missing accents, light typos.
- **StratifiedGroupKFold (5 folds) grouped by template**: the model is always tested on phrasings it never saw.

**Transparency for the README/deck:** the corpus is synthetic, generated by the team (templates drafted with Claude's help);
these are not real customer conversations. Code: `ml/intent_corpus.py`, `ml/intent_classifier.py`.

In [2]:
import sys, json
sys.path.insert(0, ".."); sys.path.insert(0, "../ml")
from intent_corpus import generar
df_synth = generar()
print(df_synth.shape, "| templates:", df_synth["template_id"].nunique())
print(df_synth.groupby(["label", "lang"]).size().unstack())
df_synth.sample(8, random_state=1)[["text", "label", "lang"]]

(1273, 4) | templates: 216
lang            es   pt
label                  
Commercial     105  107
Complaint      104  106
Product        106  105
Retention       94   95
Technical      106  106
Transactional  120  119


,text,label,lang
190,"Hola, me interesa un crédito hipotecario, que ...",Product,es
248,"Oiga, esta es la tercera vez que llamo por lo ...",Complaint,es
1238,"Oi, se nao baixarem a anuidade eu cancelo o ca...",Retention,pt
1106,quais parcelamentos sem juros tem este mes.,Commercial,pt
1272,"Boa tarde, vou para um banco digital, cancelem...",Retention,pt
653,"Boa tarde, meu saldo não reflete um deposito d...",Transactional,pt
870,o atendente foi muito mal-educado comigo na li...,Complaint,pt
556,"Oiga, estoy pensando seriamente en cambiarme d...",Retention,es


## Step 4 — Baseline (same keyword list as v1) vs. ML (TF-IDF words + characters + Logistic Regression)

In [3]:
from ml.intent_classifier import evaluar_y_exportar
res = evaluar_y_exportar()   # also exports ml/artifacts/*.joblib and web/data/intent_metrics.json
import pandas as pd
pd.DataFrame({
    "Approach": [res["baseline"]["nombre"], res["ml"]["nombre"]],
    "Macro F1": [res["baseline"]["f1_macro"], res["ml"]["f1_macro"]],
    "Accuracy": [res["baseline"]["accuracy"], res["ml"]["accuracy"]],
})

,Approach,Macro F1,Accuracy
0,Palabras clave (lista del notebook v1),0.451,0.469
1,TF-IDF palabras + caracteres + Regresion Logis...,0.887,0.887


In [4]:
pd.DataFrame({"Baseline": res["baseline"]["por_clase"], "ML": res["ml"]["por_clase"]})

,Baseline,ML
Transactional,0.416,0.926
Product,0.583,0.804
Complaint,0.279,0.941
Technical,0.687,0.966
Commercial,0.523,0.821
Retention,0.219,0.864


## Step 5 — Spanish vs. Portuguese (challenge requirement)

In [5]:
pd.DataFrame(res["por_idioma"]).T

,baseline,ml,n
es,0.425,0.885,635.0
pt,0.474,0.888,638.0


In [6]:
print("Per-fold macro F1:", res["folds"])
print("Language detector accuracy (ES/PT):", res["deteccion_idioma"]["accuracy"])
pd.DataFrame(res["confusion_ml"]["matrix"], index=res["confusion_ml"]["labels"], columns=res["confusion_ml"]["labels"])

Per-fold macro F1: [{'ml': 0.828, 'baseline': 0.36}, {'ml': 0.901, 'baseline': 0.466}, {'ml': 0.931, 'baseline': 0.397}, {'ml': 0.933, 'baseline': 0.512}, {'ml': 0.83, 'baseline': 0.479}]
Language detector accuracy (ES/PT): 0.9969


,Transactional,Product,Complaint,Technical,Commercial,Retention
Transactional,232,6,0,0,1,0
Product,0,195,0,0,8,8
Complaint,7,0,191,0,6,6
Technical,14,0,0,198,0,0
Commercial,0,51,2,0,158,1
Retention,9,22,3,0,0,155


## Conclusions

1. The dataset's real transcript text is generic boilerplate unrelated to its label — nothing to classify there.
2. Our v1 corpus was too small and its random split leaked templates: ML did not beat keywords (0.745 vs 0.749) and
   Portuguese lagged (0.657). We report it instead of hiding it.
3. With the v2 corpus and a template-grouped evaluation, **ML clearly beats the keyword baseline** (see table above) and
   performs **the same in Spanish and Portuguese** — evidence the approach is not biased toward Spanish.
4. The model and a language detector are exported to `ml/artifacts/` and used by the agent on every customer message
   (intent + language are shown in the trace/audit view).

### Limitation for the README/deck (copy as-is)

> The intent classifier is trained on our own synthetic corpus in Spanish and Portuguese (~1,270 phrases from 216
> templates), not on the dataset's transcripts — the text in `call_transcripts` is a generic template unrelated to its label.
> It is evaluated on templates never seen in training, but it is still synthetic text: real customer messages will be
> messier, and the model must be re-validated with real conversations before production.